In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("simple_tabular_training_data.csv")
print("Dataset size:", df.shape)
print(df.head())

Dataset size: (1020, 8)
        City  sensor_01  sensor_02  sensor_03  sensor_04  sensor_05  \
0  Cambridge      75.76      67.33      18.76      84.26      46.86   
1  Edinburgh      61.95      53.81      48.45      59.42      36.41   
2  Liverpool      34.36      27.82      66.48      21.71      32.50   
3  Liverpool      82.45      69.96      12.80      95.82      40.27   
4     London      46.01      35.40      47.35      48.14      39.48   

   sensor_06 Humidity_Level  
0      42.83           High  
1      42.22       Moderate  
2      36.59            Low  
3      42.40           High  
4      20.13       Moderate  


In [3]:
df = df.dropna(subset=["Humidity_Level"]).copy()
exclude = ["City", "Humidity_Level"]
X = df.drop(columns=exclude)
y = df["Humidity_Level"]
groups = df["City"]   # Only used in splitting Option B

print("\nInput columns:", X.columns.tolist())
print("\nMissing sensor values:\n", X.isna().sum())
print("\nHumidity categories:\n", y.value_counts())

# A computer works more easily with numbers than with text labels.
# LabelEncoder turns categories such as 'Low' into integer class IDs.
encoder = LabelEncoder()
y = encoder.fit_transform(y)
print("\nClass names:", encoder.classes_)


Input columns: ['sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06']

Missing sensor values:
 sensor_01    0
sensor_02    0
sensor_03    0
sensor_04    0
sensor_05    0
sensor_06    0
dtype: int64

Humidity categories:
 Humidity_Level
Moderate    388
Low         319
High        313
Name: count, dtype: int64

Class names: ['High' 'Low' 'Moderate']


In [ ]:

# A city cannot appear in more than one set. This asks a harder question:
# Can the model make predictions for cities it never saw in training?
# The class proportions are NOT guaranteed to match across sets.
#

#splitting to obtain training set and testing set
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_ids, test_ids = next(splitter.split(X, y, groups=groups))
X_train = X.iloc[train_ids]
X_test = X.iloc[test_ids]
y_train = y[train_ids]
y_test = y[test_ids]

#splittig the train set further to optain the validation set
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=21)
train_ids2, val_ids = next(
     splitter.split(X_train, y_train, groups=groups.iloc[train_ids])
)
X_val = X_train.iloc[val_ids]
y_val = y_train[val_ids]
X_train = X_train.iloc[train_ids2]
y_train = y_train[train_ids2]